# Feature Engineering for Advanced Models
Load full year data and create comprehensive features for XGBoost model

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


## Utility Functions

In [2]:
def get_black_friday(year):
    """
    Returns the Black Friday date for a given year
    Black Friday = The day after the 4th Thursday in November (Thanksgiving)
    """
    nov_first = datetime(year, 11, 1)
    days_until_thursday = (3 - nov_first.weekday()) % 7
    first_thursday = nov_first + timedelta(days=days_until_thursday)
    thanksgiving = first_thursday + timedelta(weeks=3)
    black_friday = thanksgiving + timedelta(days=1)
    return black_friday

def get_us_holidays(year):
    """
    Returns major US holidays for a given year
    """
    holidays = [
        datetime(year, 1, 1),   # New Year's Day
        datetime(year, 7, 4),   # Independence Day
        datetime(year, 12, 25), # Christmas
    ]
    
    # Thanksgiving (4th Thursday of November)
    thanksgiving = get_black_friday(year) - timedelta(days=1)
    holidays.append(thanksgiving)
    
    # Black Friday
    holidays.append(get_black_friday(year))
    
    return holidays

# Test functions
print("Black Friday 2024:", get_black_friday(2024))
print("US Holidays 2024:", get_us_holidays(2024))

Black Friday 2024: 2024-11-29 00:00:00
US Holidays 2024: [datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 7, 4, 0, 0), datetime.datetime(2024, 12, 25, 0, 0), datetime.datetime(2024, 11, 28, 0, 0), datetime.datetime(2024, 11, 29, 0, 0)]


## Load Full Year Data

In [3]:
# Load full year data for each year
data_dir = 'dataset/hrl_load_metered/'
all_years_data = []

for year in range(2016, 2025):
    print(f"\nLoading {year} data...")
    
    file_path = os.path.join(data_dir, f'hrl_load_metered_{year}.csv')
    
    if not os.path.exists(file_path):
        print(f"  File not found: {file_path}")
        continue
    
    # Read full year data
    df = pd.read_csv(file_path, usecols=['datetime_beginning_ept', 'load_area', 'mw'])
    
    # Convert datetime
    df['datetime_beginning_ept'] = pd.to_datetime(df['datetime_beginning_ept'])
    
    # Add year column
    df['year'] = year
    
    print(f"  Loaded: {len(df):,} rows")
    all_years_data.append(df)

# Combine all years
full_df = pd.concat(all_years_data, ignore_index=True)
print(f"\n{'='*80}")
print(f"Total data loaded: {len(full_df):,} rows")
print(f"Date range: {full_df['datetime_beginning_ept'].min()} to {full_df['datetime_beginning_ept'].max()}")
print(f"Load areas: {full_df['load_area'].nunique()}")
print(f"{'='*80}")


Loading 2016 data...
  Loaded: 245,952 rows

Loading 2017 data...
  Loaded: 250,417 rows

Loading 2018 data...
  Loaded: 254,784 rows

Loading 2019 data...
  Loaded: 262,800 rows

Loading 2020 data...
  Loaded: 263,520 rows

Loading 2021 data...
  Loaded: 262,800 rows

Loading 2022 data...
  Loaded: 262,800 rows

Loading 2023 data...
  Loaded: 262,800 rows

Loading 2024 data...
  Loaded: 263,520 rows

Total data loaded: 2,329,393 rows
Date range: 2016-01-01 00:00:00 to 2024-12-31 23:00:00
Load areas: 31


## Basic Time Features

In [4]:
print("Creating basic time features...")

# Time components
full_df['hour'] = full_df['datetime_beginning_ept'].dt.hour
full_df['day'] = full_df['datetime_beginning_ept'].dt.day
full_df['day_of_week'] = full_df['datetime_beginning_ept'].dt.dayofweek  # 0=Monday, 6=Sunday
full_df['day_of_year'] = full_df['datetime_beginning_ept'].dt.dayofyear
full_df['week_of_year'] = full_df['datetime_beginning_ept'].dt.isocalendar().week
full_df['month'] = full_df['datetime_beginning_ept'].dt.month
full_df['quarter'] = full_df['datetime_beginning_ept'].dt.quarter

# Weekend indicator
full_df['is_weekend'] = (full_df['day_of_week'] >= 5).astype(int)

# Time of day categories
full_df['time_of_day'] = pd.cut(full_df['hour'], 
                                  bins=[0, 6, 12, 18, 24],
                                  labels=['night', 'morning', 'afternoon', 'evening'],
                                  include_lowest=True)

# Cyclical encoding for hour and month (for models that benefit from it)
full_df['hour_sin'] = np.sin(2 * np.pi * full_df['hour'] / 24)
full_df['hour_cos'] = np.cos(2 * np.pi * full_df['hour'] / 24)
full_df['month_sin'] = np.sin(2 * np.pi * full_df['month'] / 12)
full_df['month_cos'] = np.cos(2 * np.pi * full_df['month'] / 12)
full_df['day_of_week_sin'] = np.sin(2 * np.pi * full_df['day_of_week'] / 7)
full_df['day_of_week_cos'] = np.cos(2 * np.pi * full_df['day_of_week'] / 7)

print("Basic time features created")
print(f"New columns: {list(full_df.columns[-15:])}")

Creating basic time features...
Basic time features created
New columns: ['hour', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'month', 'quarter', 'is_weekend', 'time_of_day', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos']


## Black Friday and Holiday Features

In [5]:
print("Creating Black Friday and holiday features...")

# Black Friday dates
black_fridays = {year: get_black_friday(year) for year in range(2016, 2025)}

# Add Black Friday date for each row
full_df['black_friday'] = full_df['year'].map(black_fridays)

# Days from Black Friday
full_df['days_from_bf'] = (full_df['datetime_beginning_ept'] - full_df['black_friday']).dt.days

# Is Black Friday week (7 days before to Black Friday)
full_df['is_bf_week'] = ((full_df['days_from_bf'] >= -7) & (full_df['days_from_bf'] <= 0)).astype(int)

# Thanksgiving (day before Black Friday)
full_df['is_thanksgiving'] = (full_df['days_from_bf'] == -1).astype(int)

# Holiday indicator
def is_holiday(row):
    holidays = get_us_holidays(row['year'])
    return int(row['datetime_beginning_ept'].date() in [h.date() for h in holidays])

full_df['is_holiday'] = full_df.apply(is_holiday, axis=1)

print("Black Friday and holiday features created")
print(f"Black Friday dates: {black_fridays}")

Creating Black Friday and holiday features...
Black Friday and holiday features created
Black Friday dates: {2016: datetime.datetime(2016, 11, 25, 0, 0), 2017: datetime.datetime(2017, 11, 24, 0, 0), 2018: datetime.datetime(2018, 11, 23, 0, 0), 2019: datetime.datetime(2019, 11, 29, 0, 0), 2020: datetime.datetime(2020, 11, 27, 0, 0), 2021: datetime.datetime(2021, 11, 26, 0, 0), 2022: datetime.datetime(2022, 11, 25, 0, 0), 2023: datetime.datetime(2023, 11, 24, 0, 0), 2024: datetime.datetime(2024, 11, 29, 0, 0)}


## Lag Features (Historical Values)

In [6]:
print("Creating lag features...")
print("This may take several minutes...")

# Sort by load_area and datetime for proper lag calculation
full_df = full_df.sort_values(['load_area', 'datetime_beginning_ept']).reset_index(drop=True)

# Define lag periods (in hours)
lag_hours = [1, 2, 3, 24, 48, 168]  # 1h, 2h, 3h, 1day, 2days, 1week

for lag in lag_hours:
    print(f"  Creating lag_{lag}h...")
    full_df[f'mw_lag_{lag}h'] = full_df.groupby('load_area')['mw'].shift(lag)

# Lag for same day of week, same hour (weekly pattern)
print(f"  Creating lag_1week_same_hour...")
full_df['mw_lag_1week_same_hour'] = full_df.groupby('load_area')['mw'].shift(168)

print("Lag features created")

Creating lag features...
This may take several minutes...
  Creating lag_1h...
  Creating lag_2h...
  Creating lag_3h...
  Creating lag_24h...
  Creating lag_48h...
  Creating lag_168h...
  Creating lag_1week_same_hour...
Lag features created


## Rolling Statistics

In [7]:
print("Creating rolling statistics...")
print("This may take several minutes...")

# Define rolling windows (in hours)
windows = [24, 168, 720]  # 1 day, 1 week, 30 days

for window in windows:
    window_name = f"{window}h"
    if window == 24:
        window_name = "24h"
    elif window == 168:
        window_name = "7d"
    elif window == 720:
        window_name = "30d"
    
    print(f"  Creating rolling stats for {window_name}...")
    
    # Rolling mean
    full_df[f'mw_rolling_mean_{window_name}'] = full_df.groupby('load_area')['mw'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )
    
    # Rolling std
    full_df[f'mw_rolling_std_{window_name}'] = full_df.groupby('load_area')['mw'].transform(
        lambda x: x.rolling(window=window, min_periods=1).std()
    )
    
    # Rolling min
    full_df[f'mw_rolling_min_{window_name}'] = full_df.groupby('load_area')['mw'].transform(
        lambda x: x.rolling(window=window, min_periods=1).min()
    )
    
    # Rolling max
    full_df[f'mw_rolling_max_{window_name}'] = full_df.groupby('load_area')['mw'].transform(
        lambda x: x.rolling(window=window, min_periods=1).max()
    )

print("Rolling statistics created")

Creating rolling statistics...
This may take several minutes...
  Creating rolling stats for 24h...
  Creating rolling stats for 7d...
  Creating rolling stats for 30d...
Rolling statistics created


## Data Summary

In [8]:
print("\n=== Feature Engineering Summary ===")
print(f"Total rows: {len(full_df):,}")
print(f"Total features: {len(full_df.columns)}")
print(f"\nFeature columns:")
for col in full_df.columns:
    print(f"  - {col}")

print(f"\nMissing values:")
missing = full_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    print(missing)
else:
    print("  No missing values")

print(f"\nData types:")
print(full_df.dtypes.value_counts())


=== Feature Engineering Summary ===
Total rows: 2,329,393
Total features: 43

Feature columns:
  - datetime_beginning_ept
  - load_area
  - mw
  - year
  - hour
  - day
  - day_of_week
  - day_of_year
  - week_of_year
  - month
  - quarter
  - is_weekend
  - time_of_day
  - hour_sin
  - hour_cos
  - month_sin
  - month_cos
  - day_of_week_sin
  - day_of_week_cos
  - black_friday
  - days_from_bf
  - is_bf_week
  - is_thanksgiving
  - is_holiday
  - mw_lag_1h
  - mw_lag_2h
  - mw_lag_3h
  - mw_lag_24h
  - mw_lag_48h
  - mw_lag_168h
  - mw_lag_1week_same_hour
  - mw_rolling_mean_24h
  - mw_rolling_std_24h
  - mw_rolling_min_24h
  - mw_rolling_max_24h
  - mw_rolling_mean_7d
  - mw_rolling_std_7d
  - mw_rolling_min_7d
  - mw_rolling_max_7d
  - mw_rolling_mean_30d
  - mw_rolling_std_30d
  - mw_rolling_min_30d
  - mw_rolling_max_30d

Missing values:
mw_lag_168h               5208
mw_lag_1week_same_hour    5208
mw_lag_48h                1488
mw_lag_24h                 744
mw_lag_3h          

## Sample Data Check

In [9]:
# Show sample for one area
sample_area = 'AEPIMP'
sample_data = full_df[full_df['load_area'] == sample_area].head(50)

print(f"\n=== Sample Data for {sample_area} ===")
print(sample_data[[
    'datetime_beginning_ept', 'mw', 'hour', 'day_of_week', 'is_weekend',
    'days_from_bf', 'is_bf_week', 'mw_lag_24h', 'mw_rolling_mean_24h'
]].head(20))


=== Sample Data for AEPIMP ===
       datetime_beginning_ept        mw  hour  day_of_week  is_weekend  \
157824    2016-01-01 00:00:00  2791.730     0            4           0   
157825    2016-01-01 01:00:00  2757.298     1            4           0   
157826    2016-01-01 02:00:00  2679.636     2            4           0   
157827    2016-01-01 03:00:00  2662.255     3            4           0   
157828    2016-01-01 04:00:00  2625.476     4            4           0   
157829    2016-01-01 05:00:00  2702.393     5            4           0   
157830    2016-01-01 06:00:00  2764.273     6            4           0   
157831    2016-01-01 07:00:00  2822.656     7            4           0   
157832    2016-01-01 08:00:00  2832.053     8            4           0   
157833    2016-01-01 09:00:00  2925.604     9            4           0   
157834    2016-01-01 10:00:00  2952.542    10            4           0   
157835    2016-01-01 11:00:00  3033.886    11            4           0   
157836

## Save Engineered Features

In [10]:
# Save full engineered dataset
output_file = 'dataset/preprocessed/full_year_features.csv'
print(f"Saving to {output_file}...")
full_df.to_csv(output_file, index=False)

file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"\nSaved successfully!")
print(f"File: {output_file}")
print(f"Size: {file_size_mb:.2f} MB")
print(f"Rows: {len(full_df):,}")
print(f"Columns: {len(full_df.columns)}")

Saving to dataset/preprocessed/full_year_features.csv...

Saved successfully!
File: dataset/preprocessed/full_year_features.csv
Size: 917.20 MB
Rows: 2,329,393
Columns: 43


## Optional: Save Parquet Format (Faster Loading)

In [12]:
# Parquet is more efficient for large datasets
parquet_file = 'dataset/preprocessed/full_year_features.parquet'
print(f"Saving to {parquet_file}...")
full_df.to_parquet(parquet_file, index=False)

parquet_size_mb = os.path.getsize(parquet_file) / (1024 * 1024)
print(f"\nParquet file saved!")
print(f"File: {parquet_file}")
print(f"Size: {parquet_size_mb:.2f} MB (vs CSV: {file_size_mb:.2f} MB)")
print(f"Compression ratio: {file_size_mb / parquet_size_mb:.2f}x")

Saving to dataset/preprocessed/full_year_features.parquet...

Parquet file saved!
File: dataset/preprocessed/full_year_features.parquet
Size: 225.25 MB (vs CSV: 917.20 MB)
Compression ratio: 4.07x


## Feature Importance Preview (Quick Analysis)

In [11]:
# Show correlation of features with target (mw)
numeric_cols = full_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('mw')  # Remove target

correlations = full_df[numeric_cols + ['mw']].corr()['mw'].abs().sort_values(ascending=False)

print("\n=== Top 20 Features by Correlation with MW ===")
print(correlations.head(21))  # 20 + target itself


=== Top 20 Features by Correlation with MW ===
mw                        1.000000
mw_lag_1h                 0.999227
mw_lag_2h                 0.997101
mw_lag_24h                0.996635
mw_lag_3h                 0.993996
mw_lag_48h                0.992620
mw_rolling_mean_24h       0.991304
mw_lag_1week_same_hour    0.991238
mw_lag_168h               0.991238
mw_rolling_max_24h        0.990305
mw_rolling_min_24h        0.988357
mw_rolling_mean_7d        0.987975
mw_rolling_mean_30d       0.986051
mw_rolling_max_7d         0.985486
mw_rolling_min_7d         0.985181
mw_rolling_min_30d        0.983268
mw_rolling_max_30d        0.982845
mw_rolling_std_30d        0.944636
mw_rolling_std_7d         0.932120
mw_rolling_std_24h        0.908959
hour                      0.026960
Name: mw, dtype: float64
